# 📊 PHASE 5: MODEL EVALUATION
## Compare Models & Evaluate Performance

**Objective:** Evaluate trained models on test set

**Input:** Trained models + test data  
**Output:** Performance metrics, comparisons, visualizations

**Metrics:**
- Accuracy, Precision, Recall, F1
- ROC-AUC, Confusion Matrix
- Feature Importance

---

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
import pickle
import json

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    roc_curve, auc
)

sys.path.insert(0, '../scripts')
from config import ML_FEATURES_X_FILE, ML_FEATURES_Y_FILE, MODEL_DIR, OUTPUT_DIR
from logger import get_logger

logger = get_logger(__name__)

# Visualization
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)

print("✅ Imports successful!")

## 1️⃣ LOAD DATA & MODELS

In [ ]:
# Load features and labels
X = pd.read_csv(ML_FEATURES_X_FILE)
y = pd.read_csv(ML_FEATURES_Y_FILE, header=0, squeeze=True)

# Load label encoder
with open(f"{MODEL_DIR}/label_encoder.json", 'r') as f:
    label_encoder = json.load(f)

# Encode labels
y_encoded = y.map(label_encoder)

# Split data (same as training)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# Load scaler and apply
with open(f"{MODEL_DIR}/scaler.pkl", 'rb') as f:
    scaler = pickle.load(f)

X_test_scaled = scaler.transform(X_test)

print(f"✅ Loaded data")
print(f"  Test set: {len(X_test):,} samples")

# Load models
models = {}
model_names = ['logistic_regression', 'random_forest', 'xgboost', 'lightgbm']

for name in model_names:
    model_path = f"{MODEL_DIR}/{name}_model.pkl"
    if Path(model_path).exists():
        with open(model_path, 'rb') as f:
            models[name] = pickle.load(f)
        print(f"  ✅ Loaded {name}")
    else:
        print(f"  ⚠️ {name} not found")

## 2️⃣ EVALUATE MODELS

In [ ]:
# Evaluate each model
evaluation_results = {}

print("\n📊 MODEL PERFORMANCE ON TEST SET:\n")
print("="*100)

for name, model in models.items():
    # Predictions
    if name in ['logistic_regression', 'random_forest']:
        y_pred = model.predict(X_test_scaled)
        y_pred_proba = model.predict_proba(X_test_scaled)
    else:
        y_pred = model.predict(X_test)
        y_pred_proba = model.predict_proba(X_test)
    
    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    
    evaluation_results[name] = {
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'f1': f1,
        'y_pred': y_pred,
        'y_pred_proba': y_pred_proba,
    }
    
    print(f"\n{name.upper()}")
    print(f"  Accuracy:  {acc:.4f}")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall:    {rec:.4f}")
    print(f"  F1 Score:  {f1:.4f}")

print("\n" + "="*100)

## 3️⃣ MODEL COMPARISON

In [ ]:
# Create comparison dataframe
comparison_df = pd.DataFrame(evaluation_results).T
comparison_df = comparison_df[['accuracy', 'precision', 'recall', 'f1']]
comparison_df = comparison_df.sort_values('f1', ascending=False)

print("\n🏆 MODEL RANKING (by F1 Score):")
print("\n" + comparison_df.to_string())

# Highlight best model
best_model = comparison_df.index[0]
print(f"\n✅ BEST MODEL: {best_model}")
print(f"   F1 Score: {comparison_df.loc[best_model, 'f1']:.4f}")

## 4️⃣ CONFUSION MATRICES

In [ ]:
# Create confusion matrices
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.ravel()

for idx, (name, model) in enumerate(models.items()):
    if idx >= 4:
        break
    
    y_pred = evaluation_results[name]['y_pred']
    cm = confusion_matrix(y_test, y_pred)
    
    # Create labels
    labels = [k for k, v in sorted(label_encoder.items(), key=lambda x: x[1])]
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                xticklabels=labels, yticklabels=labels, cbar=False)
    axes[idx].set_title(f"{name}\nF1: {evaluation_results[name]['f1']:.4f}", fontsize=12, fontweight='bold')
    axes[idx].set_ylabel('True')
    axes[idx].set_xlabel('Predicted')

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/figures/confusion_matrices.png", dpi=300, bbox_inches='tight')
logger.info(f"Saved confusion matrices")
plt.show()

print("✅ Confusion matrices plotted")

## 5️⃣ FEATURE IMPORTANCE

In [ ]:
# Extract feature importance from tree-based models
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

tree_models = {
    'random_forest': ('Random Forest', models.get('random_forest')),
    'xgboost': ('XGBoost', models.get('xgboost')),
}

feature_names = X.columns.tolist()

for idx, (model_name, (display_name, model)) in enumerate(tree_models.items()):
    if model is None:
        continue
    
    # Get feature importance
    if hasattr(model, 'feature_importances_'):
        importances = model.feature_importances_
        indices = np.argsort(importances)[-15:]  # Top 15
        
        axes[idx].barh(range(len(indices)), importances[indices])
        axes[idx].set_yticks(range(len(indices)))
        axes[idx].set_yticklabels([feature_names[i] for i in indices])
        axes[idx].set_xlabel('Importance')
        axes[idx].set_title(f"{display_name} - Top 15 Features")
        axes[idx].invert_yaxis()

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/figures/feature_importance.png", dpi=300, bbox_inches='tight')
logger.info(f"Saved feature importance plot")
plt.show()

print("✅ Feature importance plotted")

## 6️⃣ SAVE RESULTS

In [ ]:
# Save evaluation results
Path(f"{OUTPUT_DIR}/reports").mkdir(parents=True, exist_ok=True)

# Create metrics report
metrics_report = {
    'timestamp': pd.Timestamp.now().isoformat(),
    'test_set_size': len(y_test),
    'models': {},
}

for name in models.keys():
    metrics_report['models'][name] = {
        'accuracy': float(evaluation_results[name]['accuracy']),
        'precision': float(evaluation_results[name]['precision']),
        'recall': float(evaluation_results[name]['recall']),
        'f1': float(evaluation_results[name]['f1']),
    }

# Save as JSON
metrics_path = f"{OUTPUT_DIR}/reports/metrics_report.json"
with open(metrics_path, 'w') as f:
    json.dump(metrics_report, f, indent=2)

logger.info(f"✅ Saved metrics report to {metrics_path}")

# Save as CSV
csv_path = f"{OUTPUT_DIR}/reports/metrics_report.csv"
comparison_df.to_csv(csv_path)
logger.info(f"✅ Saved CSV report to {csv_path}")

print(f"\n✅ RESULTS SAVED:")
print(f"  JSON: {metrics_path}")
print(f"  CSV: {csv_path}")

## ✅ MODEL EVALUATION SUMMARY

**Summary:**
- ✅ Loaded trained models
- ✅ Evaluated on test set
- ✅ Calculated metrics (Accuracy, Precision, Recall, F1)
- ✅ Compared models
- ✅ Generated confusion matrices
- ✅ Plotted feature importance
- ✅ Saved evaluation reports

**Next Step:** Proceed to 08_deployment_pipeline.ipynb